# BERTTopic on Transcript Embeddings

Uses pre-computed 384-dim embeddings from the DB.
Text is filtered to lemmatised nouns only (spaCy) before topic modelling.

In [ ]:
!pip install bertopic spacy -q
!python -m spacy download en_core_web_sm -q

import sys
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import spacy
from bertopic import BERTopic
from sklearn.preprocessing import normalize

sys.path.insert(0, '.')
from models import Session, TranscriptChunk

In [ ]:
session = Session()

rows = (
    session.query(TranscriptChunk.id, TranscriptChunk.text, TranscriptChunk.embedding)
    .filter(TranscriptChunk.embedding.isnot(None))
    .all()
)

session.close()
print(f"Loaded {len(rows)} chunks")

ids        = [r.id for r in rows]
raw_texts  = [r.text or '' for r in rows]
embeddings = np.array([r.embedding for r in rows], dtype=np.float32)
embeddings = normalize(embeddings)

In [ ]:
NLP_CHAR_LIMIT = 8000

EXTRA_STOPWORDS = {
    'thing', 'stuff', 'lot', 'bit', 'way', 'time', 'day', 'year',
    'video', 'channel', 'stream', 'chat', 'comment', 'question', 'answer',
    'guy', 'man', 'woman', 'people', 'person', 'everyone', 'someone',
    'kind', 'sort', 'type', 'part', 'point', 'place', 'end', 'world',
    'sense', 'fact', 'case', 'reason', 'problem', 'issue', 'example',
    'number', 'level', 'moment', 'mind', 'word', 'life',
}

print('Loading spaCy model...')
nlp = spacy.load('en_core_web_sm', disable=['parser', 'senter'])
nlp.max_length = NLP_CHAR_LIMIT + 100


def _doc_to_nouns(doc):
    ent_spans = set()
    for ent in doc.ents:
        if ent.label_ in ('PERSON', 'ORG'):
            for tok in ent:
                ent_spans.add(tok.i)

    nouns = []
    for tok in doc:
        if tok.i in ent_spans:
            continue
        if tok.pos_ != 'NOUN':
            continue
        lemma = tok.lemma_.lower()
        if len(lemma) < 3 or lemma in EXTRA_STOPWORDS:
            continue
        nouns.append(lemma)

    return ' '.join(nouns)


PIPE_BATCH = 512  # texts per nlp.pipe batch — tune up if RAM allows
n = len(raw_texts)
clipped = [t[:NLP_CHAR_LIMIT] if isinstance(t, str) else '' for t in raw_texts]

print(f'Extracting nouns from {n} chunks via nlp.pipe (batch={PIPE_BATCH})...')
noun_docs = [
    _doc_to_nouns(doc)
    for doc in nlp.pipe(clipped, batch_size=PIPE_BATCH)
]
print('Done.')

# Drop chunks where noun extraction produced nothing
mask       = [bool(d.strip()) for d in noun_docs]
noun_docs  = [d for d, m in zip(noun_docs, mask) if m]
embeddings = embeddings[mask]
print(f"{sum(mask)} chunks with nouns / {n} total")

In [ ]:
topic_model = BERTopic(
    embedding_model=None,          # skip re-encoding
    calculate_probabilities=False,
    verbose=True,
)

topics, _ = topic_model.fit_transform(noun_docs, embeddings=embeddings)

In [ ]:
topic_info = topic_model.get_topic_info()
print(topic_info[['Topic', 'Count', 'Name']].to_string(index=False))

In [ ]:
print('\n=== TOP WORDS PER TOPIC ===\n')
for topic_id in sorted(topic_model.get_topics()):
    if topic_id == -1:
        continue
    words = [w for w, _ in topic_model.get_topic(topic_id)]
    print(f"Topic {topic_id:3d}: {', '.join(words)}")